# Lab 4: Two-Dimensional Signals — Images & Convolution

**Signal, Image and Video Processing**

### What we'll do

| Part | Topic | Level |
|------|-------|-------|
| 1 | Load, inspect, and manipulate images as arrays | 🟢 Basic |
| 2 | Histograms, equalisation, and CLAHE | 🟢 Basic |
| 3 | Convolution with kernels: blur, sharpen, edges | 🟡 Intermediate |
| 4 | The 2D Fourier Transform | 🟡 Intermediate |
| 5 | Frequency-domain filtering (LP / HP masks) | 🟡 Intermediate |
| 6 | Edge detection: Sobel and Canny | 🟡 Intermediate |
| 7 | Application: Photo 51 and the double helix | 🔴 Advanced |

### How to use this notebook

- **Read the markdown cells** before running the code — they explain *why* we do each step.
- **Run each code cell** and look at the output before moving on.
- Look for 🧪 **Experiment** boxes — these suggest things to change and re-run. Try them!
- The code is intentionally short. If something is unclear, re-read the lecture notes.

In [ ]:
# Run this cell first to load all libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data

%matplotlib inline

plt.rcParams.update({'figure.dpi': 100, 'figure.facecolor': 'white'})

---
## Part 1: Images as Arrays of Numbers 🟢

A digital image is just a 2D array (matrix) of numbers. Each element is a **pixel** — a single intensity value. For colour images, each pixel has three values (Red, Green, Blue), so the array is 3D: $(H, W, 3)$.

We'll use `skimage.data.astronaut()` as our main image — it's built into scikit-image so there's nothing to download.

In [ ]:
# Load the astronaut image (RGB, uint8)
img_rgb = data.astronaut()

print(f"Shape:  {img_rgb.shape}")    # (H, W, 3)
print(f"Dtype:  {img_rgb.dtype}")    # uint8
print(f"Range:  {img_rgb.min()} – {img_rgb.max()}")  # 0–255

plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.title(f'Colour image — shape {img_rgb.shape}')
plt.axis('off')
plt.show()

### Channels and grayscale

A colour image has three channels. Let's look at them separately, then convert to grayscale.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))

axes[0].imshow(img_rgb)
axes[0].set_title('Colour (RGB)')

for i, name in enumerate(['Red', 'Green', 'Blue']):
    axes[i+1].imshow(img_rgb[:, :, i], cmap='gray')
    axes[i+1].set_title(f'{name} channel')

# Grayscale = weighted average: 0.299R + 0.587G + 0.114B
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
axes[4].imshow(img_gray, cmap='gray')
axes[4].set_title(f'Grayscale — shape {img_gray.shape}')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Notice the red channel is brightest on the suit (which is reddish).

### The coordinate system

NumPy arrays use **row-major** order: `img[row, col]` = `img[y, x]`. The origin `(0, 0)` is at the **top-left** corner, with y increasing downward.

In [ ]:
# Access a single pixel
print(f"Pixel at row=100, col=200 (RGB): {img_rgb[100, 200]}")
print(f"Pixel at row=100, col=200 (gray): {img_gray[100, 200]}")

# Crop a region: rows 150–300, columns 200–350
crop = img_rgb[150:300, 200:350]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_rgb)
from matplotlib.patches import Rectangle
rect = Rectangle((200, 150), 150, 150, linewidth=2, edgecolor='red', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title('Original with crop region')
axes[0].axis('off')

axes[1].imshow(crop)
axes[1].set_title(f'Cropped — shape {crop.shape}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

Notice that the value for grey is not the average of red, blue and green. Investigate how it is calculated. 

🧪 **Experiment:**
- Try `img_gray[0, 0]` — what intensity is the top-left corner?
- Flip the image: `plt.imshow(img_rgb[::-1])` (vertical flip) or `img_rgb[:, ::-1]` (horizontal).
- What happens if you set `img_gray[100:200, 100:200] = 0` and display it? (Make a copy first!)

### The `uint8` trap

In [ ]:
# uint8 wraps around — this is a common source of bugs!
a = np.uint8(200)
b = np.uint8(100)
print(f"200 + 100 with uint8 = {a + b}")   # 44, not 300!
print(f"100 - 200 with uint8 = {b - a}")   # 156, not -100!

# Always convert to float before doing arithmetic
img_f = img_gray.astype(np.float64)
print(f"\nSafe: float64 range [{img_f.min():.0f}, {img_f.max():.0f}]")

---
## Part 2: Histograms, Equalisation, and CLAHE 🟢

An image **histogram** counts how many pixels have each intensity value (0–255). It tells you the *distribution* of brightnesses in the image — essential for understanding exposure problems and choosing processing strategies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Grayscale image')
axes[0].axis('off')

axes[1].hist(img_gray.ravel(), bins=256, range=(0, 256), color='gray', alpha=0.8)
axes[1].set_xlim(0, 255)
axes[1].set_xlabel('Pixel intensity')
axes[1].set_ylabel('Count')
axes[1].set_title('Histogram')

plt.tight_layout()
plt.show()

### Simulating exposure problems

Let's create dark, bright, and low-contrast versions to see how histograms diagnose problems.

In [ ]:
img_f = img_gray.astype(np.float64)
img_dark = np.clip(img_f * 0.3, 0, 255).astype(np.uint8)
img_bright = np.clip(img_f * 0.5 + 160, 0, 255).astype(np.uint8)
img_lowc = np.clip(img_f * 0.35 + 80, 0, 255).astype(np.uint8)

images = [('Dark', img_dark), ('Bright', img_bright),
          ('Low contrast', img_lowc), ('Well-exposed', img_gray)]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for j, (title, img) in enumerate(images):
    axes[0, j].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, j].set_title(title)
    axes[0, j].axis('off')
    axes[1, j].hist(img.ravel(), bins=256, range=(0, 256), color='gray', alpha=0.8)
    axes[1, j].set_xlim(0, 255)
    axes[1, j].set_xlabel('Intensity')
plt.tight_layout()
plt.show()

The histogram shape immediately reveals the problem: dark images pile up on the left, bright on the right, low contrast is a narrow spike.

### Histogram equalisation

**Histogram equalisation** remaps intensities so that the output histogram is approximately uniform — spreading the values across the full 0–255 range. It works by computing the cumulative distribution function (CDF) and using it as the mapping function.

In [ ]:
img_eq = cv2.equalizeHist(img_lowc)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].imshow(img_lowc, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Low contrast (original)')
axes[0, 0].axis('off')
axes[1, 0].hist(img_lowc.ravel(), bins=256, range=(0, 256), color='gray', alpha=0.8)
axes[1, 0].set_xlim(0, 255)

axes[0, 1].imshow(img_eq, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('After histogram equalisation')
axes[0, 1].axis('off')
axes[1, 1].hist(img_eq.ravel(), bins=256, range=(0, 256), color='gray', alpha=0.8)
axes[1, 1].set_xlim(0, 255)

plt.tight_layout()
plt.show()

### CLAHE — local, adaptive equalisation

Global equalisation applies one mapping to the whole image, which can over-amplify noise in already well-exposed regions. **CLAHE** (Contrast Limited Adaptive Histogram Equalisation) divides the image into small tiles and equalises each one independently, with a **clip limit** that prevents over-amplification.

- `tileGridSize=(8, 8)`: the image is divided into 8×8 tiles.
- `clipLimit=2.0`: any histogram bin exceeding this limit is clipped and redistributed.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img_lowc)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, title in zip(axes,
    [img_lowc, img_eq, img_clahe],
    ['Original (low contrast)', 'Global equalisation', 'CLAHE (clip=2.0, tiles=8×8)']):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

🧪 **Experiment:**
- Try different `clipLimit` values: 1.0, 5.0, 40.0. What changes?
- Try `tileGridSize=(4, 4)` vs `(16, 16)`. Smaller tiles = more local adaptation.
- Apply CLAHE to the well-exposed image. Does it help or hurt?

---
## Part 3: Convolution with Kernels 🟡

**2D convolution** slides a small matrix (the **kernel**) across the image. At each position, it computes the weighted sum of the overlapping pixels. Different kernels produce very different effects.

The formula: $(f * K)(x,y) = \sum_{i} \sum_{j} f(x-i, y-j) \cdot K(i,j)$

### Blur kernels

In [ ]:
# Box blur: every weight equal → simple average of neighbours
box_3 = cv2.blur(img_gray, (3, 3))
box_15 = cv2.blur(img_gray, (15, 15))

# Gaussian blur: weights decrease with distance from centre
gauss_3 = cv2.GaussianBlur(img_gray, (3, 3), sigmaX=1.0)
gauss_15 = cv2.GaussianBlur(img_gray, (15, 15), sigmaX=5.0)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, img, title in zip(axes,
    [img_gray, box_3, box_15, gauss_3, gauss_15],
    ['Original', 'Box 3×3', 'Box 15×15', 'Gaussian σ=1', 'Gaussian σ=5']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Blur kernels: larger kernel = stronger blur', fontsize=12)
plt.tight_layout()
plt.show()

Notice how the box blur produces visible "blocky" artefacts (especially the 15×15), while the Gaussian blur looks smoother and more natural. This is because the Gaussian weights taper off smoothly.

### Sharpening (unsharp masking)

To sharpen, we *subtract* a blurred version from the original. The difference contains the high-frequency detail (edges). Adding it back amplifies edges:

$$\text{sharpened} = \text{original} + \alpha \cdot (\text{original} - \text{blurred})$$

This can be combined into a single kernel.

In [ ]:
# Sharpening kernel (equivalent to unsharp masking with alpha=1)
kernel_sharp = np.array([[0, -1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]], dtype=np.float64)

sharpened = cv2.filter2D(img_gray, -1, kernel_sharp)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axes,
    [img_gray, gauss_3, sharpened],
    ['Original', 'Blurred (for comparison)', 'Sharpened']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Edge-detection kernels

Edge-detection kernels respond to intensity changes. The **Sobel** kernels approximate the horizontal and vertical derivatives; the **Laplacian** detects edges in all directions.

In [ ]:
# Sobel: partial derivatives
sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)  # d/dx → vertical edges
sobel_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)  # d/dy → horizontal edges
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)

# Laplacian: second derivative
laplacian = np.abs(cv2.Laplacian(img_gray, cv2.CV_64F))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, img, title, cm in zip(axes,
    [img_gray, sobel_x, sobel_y, sobel_mag, laplacian],
    ['Original', 'Sobel x (vertical edges)', 'Sobel y (horizontal edges)',
     'Sobel magnitude', 'Laplacian'],
    ['gray', 'RdBu_r', 'RdBu_r', 'gray', 'gray']):
    vmax = np.percentile(np.abs(img), 98) if title != 'Original' else None
    ax.imshow(img, cmap=cm, vmin=-vmax if 'Sobel x' in title or 'Sobel y' in title else 0, vmax=vmax)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Edge detection kernels', fontsize=12)
plt.tight_layout()
plt.show()

Notice that Sobel-x picks up vertical edges (left-right intensity changes) and Sobel-y picks up horizontal edges. The magnitude combines both.

🧪 **Experiment:**
- Try `ksize=5` in the Sobel call. How do the edges change?
- Apply Gaussian blur *before* Sobel: `cv2.GaussianBlur(img_gray, (5,5), 1.5)`. Are the edges cleaner?
- Try your own custom kernel with `cv2.filter2D`. For instance, what does `np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]])` do?

### Kernel size and separability

In [ ]:
# The Gaussian is separable: the 2D kernel = outer product of two 1D kernels
k1d = cv2.getGaussianKernel(5, 1.0)  # 1D Gaussian, size 5
k2d = k1d @ k1d.T                     # outer product → 2D kernel

print("1D kernel (column vector):")
print(np.round(k1d.ravel(), 4))
print("\n2D kernel (outer product):")
print(np.round(k2d, 4))

# Sobel-x is also separable: [1,2,1]^T × [-1,0,1]
smooth = np.array([[1], [2], [1]])
diff = np.array([[-1, 0, 1]])
print("\nSobel-x = smooth × diff:")
print(smooth @ diff)

---
## Part 4: The 2D Fourier Transform 🟡

In Lecture 2, the 1D DFT decomposed a signal into sinusoidal components at different frequencies. The **2D DFT** does the same thing with 2D sinusoidal patterns — waves that vary across the image plane.

$$F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y) \, e^{-i\,2\pi\left(\frac{ux}{M} + \frac{vy}{N}\right)}$$

Each pair $(u,v)$ specifies a spatial frequency and orientation. The coefficient $F(u,v)$ tells you how much of that pattern is present in the image.

### 2D basis patterns

Let's visualise what these basis patterns look like.

In [ ]:
N = 256
x = np.arange(N)
X, Y = np.meshgrid(x, x)

fig, axes = plt.subplots(3, 5, figsize=(14, 8))
freqs = [(0,0), (1,0), (0,1), (1,1), (3,3),
         (0,5), (5,0), (5,5), (10,0), (0,10),
         (10,10), (20,0), (0,20), (15,5), (5,15)]

for idx, (u, v) in enumerate(freqs):
    row, col = idx // 5, idx % 5
    pattern = np.cos(2 * np.pi * (u * X / N + v * Y / N))
    axes[row, col].imshow(pattern, cmap='gray', vmin=-1, vmax=1, interpolation='bilinear')
    axes[row, col].set_title(f'u={u}, v={v}', fontsize=8)
    axes[row, col].axis('off')

plt.suptitle('2D Fourier basis patterns — cosine waves at different spatial frequencies', fontsize=12)
plt.tight_layout()
plt.show()

Low frequencies (small $u$, $v$) vary slowly across the image — they represent smooth regions. High frequencies vary rapidly — they represent edges and fine detail.

### Computing and reading the 2D spectrum

The **magnitude spectrum** $|F(u,v)|$ is displayed with the zero-frequency (**DC**) component at the centre using `np.fft.fftshift`. DC stands for "Direct Current" (from electrical engineering) — it's simply the sum of all pixel values, i.e., the overall brightness.

In [ ]:
F = np.fft.fft2(img_gray.astype(np.float64))
F_shifted = np.fft.fftshift(F)
magnitude = np.log1p(np.abs(F_shifted))  # log scale for visibility

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Image (spatial domain)')
axes[0].axis('off')

im = axes[1].imshow(magnitude, cmap='gray')
axes[1].set_title('2D Fourier magnitude spectrum (log scale)')
axes[1].axis('off')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

How to read the spectrum:
- **Bright centre:** the DC component (overall brightness) is always the strongest.
- **Cross-shaped pattern:** indicates strong horizontal and vertical structures in the image.
- **Bright spots off-centre:** periodic patterns (textures, grids) at specific frequencies.
- **Spread of energy:** a sharp, detailed image has more energy at high frequencies (far from centre).

🧪 **Experiment:**
- Compute the spectrum of a *blurred* image (`cv2.GaussianBlur(img_gray, (31,31), 5)`). How does it differ? (Hint: less high-frequency energy.)
- Try `data.checkerboard()` — its spectrum should show very clear peaks corresponding to the check pattern.
- Try the spectrum of a purely vertical stripe pattern: `np.tile([0,0,0,0,255,255,255,255], (256, 32)).astype(np.uint8)`

---
## Part 5: Frequency-Domain Filtering 🟡

Just as in Lab 2 (1D signals), we can filter images by masking in the frequency domain:

1. **FFT**: compute the 2D Fourier transform
2. **Mask**: multiply by a 2D mask (ones = keep, zeros = remove)
3. **Inverse FFT**: go back to the spatial domain

A **circular** mask centred on the DC component acts as a low-pass (keep inside) or high-pass (keep outside) filter.

In [ ]:
rows, cols = img_gray.shape
crow, ccol = rows // 2, cols // 2

# Distance from centre for every pixel in the spectrum
Y_grid, X_grid = np.ogrid[:rows, :cols]
dist = np.sqrt((X_grid - ccol)**2 + (Y_grid - crow)**2)

# Low-pass: keep only low frequencies (smooth regions)
radius_lp = 30
lp_mask = (dist <= radius_lp).astype(np.float64)

# High-pass: keep only high frequencies (edges)
hp_mask = 1.0 - lp_mask

# Apply
F_shifted = np.fft.fftshift(np.fft.fft2(img_gray.astype(np.float64)))
img_lp = np.fft.ifft2(np.fft.ifftshift(F_shifted * lp_mask)).real
img_hp = np.fft.ifft2(np.fft.ifftshift(F_shifted * hp_mask)).real

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Top row: masks and results
axes[0, 0].imshow(img_gray, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 1].imshow(np.clip(img_lp, 0, 255), cmap='gray')
axes[0, 1].set_title(f'Low-pass (r={radius_lp})')
axes[0, 2].imshow(np.clip(np.abs(img_hp) * 3, 0, 255), cmap='gray')
axes[0, 2].set_title('High-pass (enhanced ×3)')

# Bottom row: spectra
axes[1, 0].imshow(np.log1p(np.abs(F_shifted)), cmap='gray')
axes[1, 0].set_title('Original spectrum')
axes[1, 1].imshow(lp_mask, cmap='gray')
axes[1, 1].set_title('Low-pass mask')
axes[1, 2].imshow(hp_mask, cmap='gray')
axes[1, 2].set_title('High-pass mask')

for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Frequency-domain filtering: LP blurs, HP extracts edges', fontsize=12)
plt.tight_layout()
plt.show()

The low-pass result is blurred (only smooth, slowly-varying content survives). The high-pass result shows only edges and sharp detail — exactly what a Sobel filter extracts in the spatial domain. This is the **convolution theorem** in action: convolution with a blur kernel in space = multiplication with a low-pass mask in frequency.

🧪 **Experiment:**
- Try different radii: 10, 50, 100. How does the blur change?
- Create a **band-pass** filter: `bp_mask = ((dist >= 20) & (dist <= 80)).astype(np.float64)`. What does it keep?
- Create a **notch filter** that removes a specific frequency (useful for removing periodic noise, which we'll explore in the lab exercises).

---
## Part 6: Edge Detection — Sobel and Canny 🟡

An **edge** is a rapid change in intensity. We already saw Sobel kernels in Part 3. Now let's compare raw Sobel with the full **Canny** edge detector, which adds three extra steps to produce clean, thin edges:

1. **Gaussian blur** — reduces noise before computing derivatives
2. **Non-maximum suppression** — thins edges by keeping only the local maximum along the gradient direction
3. **Hysteresis thresholding** — uses two thresholds to connect strong edges through weak ones

### Why blur before detecting edges?

Derivatives amplify noise. Let's see this clearly.

In [ ]:
# Sobel on raw image vs. Sobel on blurred image
Gx_raw = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
Gy_raw = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
mag_raw = np.sqrt(Gx_raw**2 + Gy_raw**2)

blurred = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1.5)
Gx_blur = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
Gy_blur = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
mag_blur = np.sqrt(Gx_blur**2 + Gy_blur**2)

# Canny (does its own internal blur)
canny = cv2.Canny(img_gray, threshold1=50, threshold2=150)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
vmax = np.percentile(mag_raw, 98)

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(mag_raw, cmap='gray', vmin=0, vmax=vmax)
axes[1].set_title('Sobel (no blur) — noisy')
axes[2].imshow(mag_blur, cmap='gray', vmin=0, vmax=vmax)
axes[2].set_title('Sobel (after blur) — cleaner')
axes[3].imshow(canny, cmap='gray')
axes[3].set_title('Canny — thin, connected')

for ax in axes:
    ax.axis('off')
plt.suptitle('Edge detection: raw Sobel vs. blurred Sobel vs. Canny', fontsize=12)
plt.tight_layout()
plt.show()

The raw Sobel output is thick and noisy. Blurring first helps, but the edges are still thick. Canny produces clean, one-pixel-wide, connected edges.

### Canny's two thresholds

The `threshold1` and `threshold2` parameters control the hysteresis:
- Pixels with gradient above `threshold2` are **definitely** edges.
- Pixels between `threshold1` and `threshold2` are edges **only if connected** to a strong edge.
- Pixels below `threshold1` are discarded.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

thresholds = [(10, 50), (50, 150), (100, 200), (150, 250)]
for ax, (t1, t2) in zip(axes, thresholds):
    edges = cv2.Canny(img_gray, t1, t2)
    ax.imshow(edges, cmap='gray')
    ax.set_title(f'Canny ({t1}, {t2})')
    ax.axis('off')

plt.suptitle('Canny with different threshold pairs', fontsize=12)
plt.tight_layout()
plt.show()

🧪 **Experiment:**
- Low thresholds (10, 50) detect many edges including noise. High thresholds (150, 250) keep only the strongest.
- Try Canny on a blurred image: `cv2.Canny(cv2.GaussianBlur(img_gray, (7,7), 2), 50, 150)`. Fewer spurious edges.
- Apply Canny to `data.coins()` — a classic segmentation test image.

---
## Part 7: Application — Photo 51 and the Double Helix 🔴

In Lecture 2, we mentioned that Rosalind Franklin's famous "Photo 51" is essentially a 2D Fourier transform. Now we can understand why.

**X-ray crystallography** works by shining X-rays through a crystal and recording the **diffraction pattern** on film. The physics of diffraction means that the recorded pattern is (approximately) the **2D Fourier transform** of the molecule's electron density.

A **helix** viewed from the side projects to a **sinusoidal curve**. A **double helix** (two strands offset by half a turn) produces a characteristic **X-shaped pattern** in Fourier space — exactly what Franklin observed.

Let's build a synthetic double helix and compute its FFT.

In [ ]:
size = 512
img_helix = np.zeros((size, size), dtype=np.float64)

cx = size // 2
pitch = 80        # pixels per full turn
radius = 60       # amplitude of the sinusoid
sigma = 5.0       # strand width

x_all = np.arange(size)

for y in range(size):
    angle = 2 * np.pi * y / pitch
    # Strand 1
    x1 = cx + radius * np.cos(angle)
    # Strand 2 (offset by pi = double helix)
    x2 = cx + radius * np.cos(angle + np.pi)
    # Each strand as a Gaussian profile
    img_helix[y, :] += np.exp(-0.5 * ((x_all - x1) / sigma)**2)
    img_helix[y, :] += np.exp(-0.5 * ((x_all - x2) / sigma)**2)

img_helix /= img_helix.max()

# 2D FFT
F_helix = np.fft.fft2(img_helix)
F_helix_shifted = np.fft.fftshift(F_helix)
mag_helix = np.log1p(np.abs(F_helix_shifted))

# Crop to interesting central region
crop = 160
c = size // 2
mag_crop = mag_helix[c-crop:c+crop, c-crop:c+crop]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_helix, cmap='gray')
axes[0].set_title('Projected double helix', fontsize=12)
axes[0].axis('off')

axes[1].imshow(mag_crop, cmap='inferno')
axes[1].set_title('2D FFT — X-shaped layer lines', fontsize=12)
axes[1].axis('off')

plt.suptitle('Photo 51: a helix projects to a sinusoid → its FT shows the characteristic X',
             fontsize=13)
plt.tight_layout()
plt.show()

## What you can read from the diffraction pattern:

- **The X-shaped cross** of bright spots: the signature of a helical structure.
- **Spacing between layer lines**: inversely proportional to the helix pitch. Closer lines = larger pitch.
- **Missing 4th layer line**: the hallmark of a *double* helix (two strands offset by half a turn cause destructive interference at every other layer line).

This is the convolution theorem at work in nature: the physical structure is "convolved" with the X-ray beam, and the detector records the Fourier transform.

🧪 **Experiment:**
- Change `pitch` to 40 or 120. How does the layer-line spacing change? (Hint: pitch and spacing are inversely related.)
- Remove strand 2 (comment out the `x2` line or set it to 0) to get a single helix. Compare the FFT — the missing layer line should reappear.
- Change `radius` — does it affect the X angle?

---
## Summary

| Concept | What it does | Key function / tool |
|---------|-------------|---------------------|
| **Image as array** | $H \times W$ matrix (grayscale) or $H \times W \times 3$ (colour) | `img.shape`, `img.dtype` |
| **Histogram** | Distribution of pixel intensities | `np.histogram()` |
| **Histogram equalisation** | Spreads intensities to use full range | `cv2.equalizeHist()` |
| **CLAHE** | Local adaptive equalisation with clip limit | `cv2.createCLAHE()` |
| **Convolution** | Slides a kernel over the image (weighted sum) | `cv2.filter2D()`, `cv2.GaussianBlur()` |
| **Sobel** | Approximates image gradients (edge detection) | `cv2.Sobel()` |
| **Canny** | Clean, thin edges via blur → gradient → NMS → hysteresis | `cv2.Canny()` |
| **2D FFT** | Decomposes image into 2D spatial frequencies | `np.fft.fft2()` |
| **Frequency mask** | LP/HP filtering by masking the 2D spectrum | `F_shifted * mask` |

**Key insight:** Convolution in space = multiplication in frequency. A blur kernel is a low-pass filter; an edge kernel is a high-pass filter. This is the **convolution theorem**.